# GLEE Competition agent V4 — history-calibrated adaptive agent

V4 is a targeted successor to V3, built from the live `llms.txt` schemas and the observed 66-game V3 history. It preserves V3's validated dispatcher and strong buyer behavior while fixing three concrete failures:

1. **Bargaining:** the V3 prior capped the responder near 50% even when visible discount factors implied an extreme equilibrium share; it also had no escape from repeated unknown-horizon cycles.
2. **Negotiation:** individually irrational agreements remain forbidden, but repeated infeasible price cycles now terminate safely and hidden-value buyer concessions use the opponent's observed trend.
3. **Persuasion:** seller policy now adapts even when buyer values are hidden, and persistent memory separates an opponent's seller reliability from its buyer responsiveness.

No notebook can guarantee a higher rating because configurations, roles, opponents, and rating updates are stochastic. Evaluate V4 in bounded, family-specific batches against V3 before replacing a working policy.


In [ ]:
%pip install -q -U glee-sdk



## API key, imports, and thread-safe memory

Named opponents get cross-game profiles. Hidden opponents get game-local memory.
Stable hashing makes mixed strategies reproducible and concurrency-safe.



In [ ]:
import hashlib
import math
import os
import statistics
import threading
from collections import defaultdict, deque
from getpass import getpass

os.environ["GLEE_API_KEY"] = getpass("GLEE API key: ")

LOCK = threading.RLock()
SEEN = set()
DECISION_LOG = deque(maxlen=1500)

BARGAINING_MEMORY = defaultdict(lambda: {
    "rejected_floor": 0.0, "opponent_demands": deque(maxlen=40)
})
NEGOTIATION_MEMORY = defaultdict(lambda: {
    "seller_prices": deque(maxlen=50), "buyer_prices": deque(maxlen=50)
})
# Perspective is part of the key. A named opponent's behavior as seller must not
# be contaminated by observations collected while that opponent was the buyer.
PERSUASION_MEMORY = defaultdict(lambda: {
    "pos_high": 0.0, "pos_low": 0.0,
    "neg_high": 0.0, "neg_low": 0.0,
    "positive_buys": 0.0, "positive_decisions": 0.0,
})


## Shared schema helpers



In [ ]:
def clamp(x, low, high):
    return max(low, min(high, x))

def finite_float(value, default=0.0):
    try:
        number = float(value)
        return number if math.isfinite(number) else default
    except (TypeError, ValueError):
        return default

def round_progress(state):
    current = max(1, int(state.get("round", 1)))
    maximum = state.get("max_rounds")
    if state.get("horizon_known") and maximum:
        return clamp((current - 1) / max(1, int(maximum) - 1), 0.0, 1.0)
    # Unknown horizons concede slowly unless explicit repetition is detected.
    return min(0.65, (current - 1) / 16.0)

def final_round(state):
    return bool(state.get("horizon_known") and state.get("max_rounds") and
                int(state.get("round", 1)) >= int(state["max_rounds"]))

def player_index(player):
    return 1 if player in {"player_1", "alice"} else 2

def canonical_player(player):
    return f"player_{player_index(player)}"

def other_player(player):
    return "player_2" if player_index(player) == 1 else "player_1"

def opponent_key(game, family):
    opponent = game.get("opponent") or {}
    if opponent.get("type") != "hidden" and opponent.get("name"):
        return f"{family}:named:{opponent.get('type')}:{opponent['name']}"
    return f"{family}:game:{game.get('game_id', '')}"

def stable_unit(game, salt=""):
    state = game.get("game_state") or {}
    token = f"{game.get('game_id', '')}:{state.get('round', 1)}:{salt}"
    value = int.from_bytes(hashlib.sha256(token.encode()).digest()[:8], "big")
    return value / 2**64

def logistic(x):
    return 1.0 / (1.0 + math.exp(-clamp(x, -60.0, 60.0)))

def allocation(offer, player):
    keys = (("player_1_gain", "alice_gain") if player_index(player) == 1 else
            ("player_2_gain", "bob_gain"))
    for key in keys:
        if key in offer:
            return finite_float(offer[key], None)
    return None

def action_message(text):
    return str(text)[:2000]

def signal_polarity(value):
    if isinstance(value, dict):
        value = value.get("decision", value.get("message"))
    if value is None:
        return None
    text = str(value).strip().lower().replace("_", " ")
    negatives = (
        "do not buy", "don't buy", "do not recommend", "don't recommend",
        "not worth", "pass", "avoid", "low quality", "bad product",
    )
    positives = (
        "buy", "recommend", "worth it", "high quality", "great product",
        "good product", "positive",
    )
    if text in {"no", "false", "negative", "not recommended"}:
        return False
    if any(phrase in text for phrase in negatives):
        return False
    if text in {"yes", "true", "recommended"}:
        return True
    if any(phrase in text for phrase in positives):
        return True
    return None

def tail_repeat_count(values, relative_tolerance=0.002):
    if not values:
        return 0
    last = float(values[-1])
    tolerance = max(1e-9, abs(last) * relative_tolerance)
    count = 0
    for value in reversed(values):
        if abs(float(value) - last) <= tolerance:
            count += 1
        else:
            break
    return count


## 1. Bargaining — uncapped equilibrium and cycle escape

V4 retains probability-weighted payoff search but no longer clips a visible Rubinstein responder share to 50%. It learns rejected shares and opponent demands up to 99.9%, expands the offer grid for extreme configurations, and detects repeated two-sided demands. After three stagnant cycles it either matches the revealed demand as proposer or accepts any strictly positive allocation as responder, preventing a repeat of V3's 99-round zero-payoff loop.


In [ ]:
def player_delta(state, player, default=0.93):
    return clamp(finite_float(state.get(f"delta_{player_index(player)}", default), default),
                 0.001, 0.9999)

def rubinstein_responder_share(proposer_delta, responder_delta):
    denominator = 1.0 - proposer_delta * responder_delta
    proposer_share = ((1.0 - responder_delta) / denominator
                      if denominator > 1e-12 else 0.5)
    return clamp(1.0 - proposer_share, 0.001, 0.999)

def bargaining_offer_series(state, money):
    series = {"player_1": [], "player_2": []}
    if money <= 0:
        return series
    for record in state.get("history", []):
        if not isinstance(record, dict):
            continue
        offer = record.get("offer") or {}
        proposer = canonical_player(record.get("proposer", offer.get("proposer", "player_1")))
        demand = allocation(offer, proposer)
        if demand is not None:
            series[proposer].append(clamp(demand / money, 0.0, 1.0))
    return series

def bargaining_stall_count(state, money):
    series = bargaining_offer_series(state, money)
    return min(tail_repeat_count(series["player_1"], 0.003),
               tail_repeat_count(series["player_2"], 0.003))

def update_bargaining_memory(game, me, opponent, money):
    key = opponent_key(game, "bargaining")
    with LOCK:
        model = BARGAINING_MEMORY[key]
        for record in game["game_state"].get("history", []):
            if not isinstance(record, dict):
                continue
            offer = record.get("offer") or {}
            proposer = canonical_player(record.get("proposer", offer.get("proposer", "player_1")))
            decision = record.get("decision")
            if isinstance(decision, dict):
                decision = decision.get("decision")
            event = ("bargaining-v4", game.get("game_id"), record.get("round"),
                     proposer, str(decision),
                     tuple(sorted((str(k), str(v)) for k, v in offer.items())))
            if event in SEEN:
                continue
            SEEN.add(event)
            if proposer == canonical_player(me) and str(decision).lower() == "reject":
                rejected = allocation(offer, opponent)
                if rejected is not None and money > 0:
                    model["rejected_floor"] = max(model["rejected_floor"], rejected / money)
            if proposer == canonical_player(opponent):
                demand = allocation(offer, opponent)
                if demand is not None and money > 0:
                    model["opponent_demands"].append(clamp(demand / money, 0.0, 1.0))
        return {"rejected_floor": model["rejected_floor"],
                "opponent_demands": list(model["opponent_demands"])}

def estimate_bargaining_floor(game, state, me, opponent, model):
    t = round_progress(state)
    if state.get("complete_information"):
        prior = rubinstein_responder_share(player_delta(state, me),
                                           player_delta(state, opponent))
    else:
        opponent_type = (game.get("opponent") or {}).get("type")
        prior = (0.46 if opponent_type == "human" else 0.43) + 0.02 * t
    evidence = model["rejected_floor"] + 0.006 if model["rejected_floor"] else 0.0
    if model["opponent_demands"]:
        recent = model["opponent_demands"][-5:]
        demand_floor = statistics.median(recent) - (0.065 - 0.020 * t)
        evidence = max(evidence, demand_floor)
    return clamp(max(prior, evidence), 0.20, 0.999)

def bargaining_strategy(game):
    state = game["game_state"]
    me = canonical_player(game.get("your_player", state["current_player"]))
    opponent = other_player(me)
    money = finite_float(state["money_to_divide"])
    t = round_progress(state)
    my_delta = player_delta(state, me)
    model = update_bargaining_memory(game, me, opponent, money)
    floor = estimate_bargaining_floor(game, state, me, opponent, model)
    stalls = bargaining_stall_count(state, money)

    if game["valid_actions"]["type"] == "offer":
        if stalls >= 3 and model["opponent_demands"]:
            # Match the opponent's revealed demand rather than repeating a split
            # they have already rejected indefinitely.
            responder_share = clamp(model["opponent_demands"][-1], 0.20, 0.9999)
        else:
            best = None
            failure_cost = 0.04 + 0.24 * t + 0.55 * (1.0 - my_delta)
            # 10% through 99.5% in half-percentage-point increments.
            for step in range(20, 200):
                responder_share = step / 200.0
                width = 0.012 if floor > 0.80 else 0.022
                probability = logistic((responder_share - floor + 0.008) / width)
                own_share = 1.0 - responder_share
                objective = probability * own_share**1.15 - (1.0 - probability) * failure_cost
                candidate = (objective, own_share, responder_share)
                if best is None or candidate > best:
                    best = candidate
            responder_share = best[2]
        responder_gain = round(money * responder_share, 8)
        own_gain = money - responder_gain
        action = ({"alice_gain": own_gain, "bob_gain": responder_gain}
                  if player_index(me) == 1 else
                  {"alice_gain": responder_gain, "bob_gain": own_gain})
        if state.get("messages_allowed"):
            pct = round(100 * responder_share, 1)
            action["message"] = action_message(
                f"I offer you {pct}% now, accounting for discounting and the observed negotiation path."
            )
        return action

    current_gain = allocation(state.get("last_offer") or {}, me)
    if current_gain is None:
        return {"decision": "reject"}
    if final_round(state):
        return {"decision": "accept" if current_gain >= 0 else "reject"}
    if stalls >= 3 and current_gain > 0:
        return {"decision": "accept"}
    next_own_share = 1.0 - floor
    deal_probability = clamp(0.80 + 0.10 * t - 0.20 * model["rejected_floor"], 0.45, 0.92)
    continuation_share = my_delta * next_own_share * deal_probability
    risk_floor = max(0.0, 0.34 - 0.08 * t - 0.07 * stalls)
    required = money * max(risk_floor, continuation_share)
    return {"decision": "accept" if current_gain + 1e-9 >= required else "reject"}


## 2. Negotiation — trend-aware surplus capture and stall termination

V4 preserves individual rationality: it never accepts negative surplus merely to increase agreement rate. It uses the opponent's latest price and concession trend inside the revealed feasible interval, with a slightly less aggressive hidden-value buyer claim than seller claim. Repeated infeasible unknown-horizon cycles end with `WalkAway`, avoiding 99-turn polling risk without sacrificing payoff.


In [ ]:
def offer_sender(item, record, field):
    sender = item.get("from_player") if isinstance(item, dict) else None
    if sender:
        return canonical_player(sender)
    if field == "counteroffer" and record.get("decided_by"):
        return canonical_player(record["decided_by"])
    return None

def negotiation_price_series(state):
    series = {"player_1": [], "player_2": []}
    for record in state.get("history", []):
        if not isinstance(record, dict):
            continue
        for field in ("offer", "counteroffer"):
            item = record.get(field)
            if isinstance(item, dict) and item.get("price") is not None:
                sender = offer_sender(item, record, field)
                if sender:
                    series[sender].append(finite_float(item["price"]))
    return series

def negotiation_stall_count(state, me, opponent):
    series = negotiation_price_series(state)
    return min(tail_repeat_count(series[canonical_player(me)], 0.001),
               tail_repeat_count(series[canonical_player(opponent)], 0.001))

def update_negotiation_memory(game, state):
    key = opponent_key(game, "negotiation")
    with LOCK:
        model = NEGOTIATION_MEMORY[key]
        for record in state.get("history", []):
            if not isinstance(record, dict):
                continue
            for field in ("offer", "counteroffer"):
                item = record.get(field)
                if not isinstance(item, dict) or item.get("price") is None:
                    continue
                sender = offer_sender(item, record, field)
                if sender is None:
                    continue
                price = finite_float(item["price"])
                event = ("negotiation-v4", game.get("game_id"), record.get("round"),
                         field, sender, price)
                if event in SEEN:
                    continue
                SEEN.add(event)
                role = state.get(f"{sender}_role")
                if role in {"seller", "buyer"}:
                    model[f"{role}_prices"].append(price)
        return {name: list(values) for name, values in model.items()}

def opponent_prices_in_game(state, opponent):
    return negotiation_price_series(state)[canonical_player(opponent)]

def projected_opponent_price(prices, role):
    if not prices:
        return None
    latest = prices[-1]
    if len(prices) < 2:
        return latest
    step = latest - prices[-2]
    # Only extrapolate concessions in the economically expected direction.
    if role == "seller":
        step = min(0.0, step)
    else:
        step = max(0.0, step)
    return max(0.0, latest + 0.6 * step)

def negotiation_strategy(game):
    state = game["game_state"]
    me = canonical_player(game.get("your_player", state["current_player"]))
    opponent = other_player(me)
    role = state[f"{me}_role"]
    opponent_role = state[f"{opponent}_role"]
    my_value = finite_float(state[f"{me}_value"])
    t = round_progress(state)
    update_negotiation_memory(game, state)
    observed = opponent_prices_in_game(state, opponent)
    stalls = negotiation_stall_count(state, me, opponent)
    opponent_value = state.get(f"{opponent}_value")
    surplus = None

    if state.get("complete_information") and opponent_value is not None:
        opponent_value = finite_float(opponent_value)
        seller_value = my_value if role == "seller" else opponent_value
        buyer_value = my_value if role == "buyer" else opponent_value
        surplus = buyer_value - seller_value
        own_capture = 0.74 - 0.14 * t
        if observed and surplus > 1e-12:
            last_price = observed[-1]
            opponent_demand = ((last_price - seller_value) / surplus if opponent_role == "seller"
                               else (buyer_value - last_price) / surplus)
            feasible_capture = 1.0 - clamp(opponent_demand - (0.05 + 0.04 * t), 0.0, 1.0)
            own_capture = 0.58 * own_capture + 0.42 * feasible_capture
        own_capture = clamp(own_capture, 0.52, 0.82)
        if surplus <= 0:
            target = my_value
        elif role == "seller":
            target = seller_value + own_capture * surplus
        else:
            target = buyer_value - own_capture * surplus
    elif observed:
        anchor = projected_opponent_price(observed, opponent_role)
        claim = ((0.70 - 0.12 * t) if role == "seller" else
                 (0.62 - 0.10 * t))
        if role == "seller" and anchor >= my_value:
            target = my_value + claim * (anchor - my_value)
        elif role == "buyer" and anchor <= my_value:
            target = my_value - claim * (my_value - anchor)
        else:
            target = my_value
    elif role == "seller":
        target = my_value * (1.36 - 0.16 * t)
    else:
        target = my_value * (0.76 + 0.12 * t)

    target = max(0.0, finite_float(target, my_value))
    if game["valid_actions"]["type"] == "offer":
        action = {"product_price": round(target, 8)}
        if state.get("messages_allowed"):
            action["message"] = action_message(
                "This price is inside the feasible interval revealed by our offers."
            )
        return action

    price = finite_float((state.get("last_offer") or {}).get("price"), my_value)
    offered_utility = price - my_value if role == "seller" else my_value - price
    profitable = offered_utility >= -1e-9
    if final_round(state):
        return {"decision": "AcceptOffer" if profitable else "RejectOffer"}
    if profitable and stalls >= 2:
        return {"decision": "AcceptOffer"}
    if not profitable and stalls >= 3 and not state.get("horizon_known"):
        return {"decision": "WalkAway"}

    target_utility = abs(target - my_value)
    offered_capture = offered_utility / surplus if surplus is not None and surplus > 1e-12 else None
    continuation = target_utility * (0.80 - 0.12 * t - 0.08 * min(stalls, 2))
    if profitable and (offered_utility + 1e-9 >= continuation or
                       (offered_capture is not None and offered_capture >= 0.50 + 0.05 * (1 - t))):
        return {"decision": "AcceptOffer"}

    blend = 0.24 + 0.43 * t + 0.08 * min(stalls, 2)
    blend = clamp(blend, 0.0, 0.78)
    counter = (1.0 - blend) * target + blend * price
    counter = max(my_value, counter) if role == "seller" else min(my_value, counter)
    action = {"decision": "RejectOffer", "product_price": round(max(0.0, counter), 8)}
    if state.get("messages_allowed"):
        action["message"] = action_message(
            "I am conceding inside my individually rational range."
        )
    return action


## 3. Persuasion — role-separated memory and adaptive seller exploration

V3's buyer earned a positive mean live rating change, so V4 preserves its direct precision estimator. Seller performance was weaker. V4 separates seller-reliability memory from buyer-responsiveness memory, increases known-value pooling only when credibility supports it, and introduces cautious late low-quality exploration when buyer values are hidden and positive recommendations are actually being purchased.


In [ ]:
def update_persuasion_memory(game, seller_view):
    state = game["game_state"]
    perspective = "buyer-response" if seller_view else "seller-reliability"
    key = opponent_key(game, f"persuasion:{perspective}")
    with LOCK:
        model = PERSUASION_MEMORY[key]
        for record in state.get("history", []):
            if not isinstance(record, dict):
                continue
            signal = signal_polarity(record.get("seller_message"))
            quality = record.get("quality")
            decision = record.get("buyer_decision")
            if isinstance(decision, dict):
                decision = decision.get("decision")
            event = ("persuasion-v4", perspective, game.get("game_id"),
                     record.get("round"), signal, quality, str(decision))
            if event in SEEN:
                continue
            SEEN.add(event)
            if signal is not None and quality in {"high", "low"}:
                model[f"{'pos' if signal else 'neg'}_{quality}"] += 1.0
            if seller_view and signal is True and str(decision).lower() in {"yes", "no"}:
                model["positive_decisions"] += 1.0
                model["positive_buys"] += float(str(decision).lower() == "yes")
        return dict(model)

def smoothed_signal_precision(model, positive, p):
    if positive:
        high, low = model["pos_high"], model["pos_low"]
        q_high, q_low = 0.90, 0.24
    else:
        high, low = model["neg_high"], model["neg_low"]
        q_high, q_low = 0.10, 0.76
    denominator = p * q_high + (1.0 - p) * q_low
    strategic_prior = p * q_high / denominator if denominator > 1e-12 else p
    strength = 4.0
    return (strength * strategic_prior + high) / (strength + high + low)

def seller_low_pool_probability(game, state, model, p, price, t, round_no, total_rounds):
    response_rate = ((model["positive_buys"] + 1.5) /
                     (model["positive_decisions"] + 2.0))
    if model["positive_decisions"] >= 3 and response_rate < 0.28:
        return 0.0
    truthful_prefix_done = round_no > max(1, math.ceil(0.10 * total_rounds))
    if not truthful_prefix_done:
        return 0.0

    if "v" in state and "u" in state:
        v, u = finite_float(state["v"]), finite_float(state["u"])
        if price <= u:
            return 1.0
        if price >= v or not (0.0 < p < 1.0) or v <= u:
            return 0.0
        cutoff = clamp((price - u) / (v - u), 1e-6, 1.0 - 1e-6)
        static_pool = p * (1.0 - cutoff) / (cutoff * (1.0 - p))
        prior_high, prior_low = 2.5, 0.8
        precision_after = ((prior_high + model["pos_high"]) /
                           (prior_high + prior_low + model["pos_high"] +
                            model["pos_low"] + 1.0))
        safety = 0.025 * (1.0 - t)
        if precision_after < cutoff + safety:
            return 0.0
        ramp = 0.28 + 0.72 * t**1.35
        return clamp(static_pool * ramp * (0.60 + 0.55 * response_rate), 0.0, 1.0)

    # When values are hidden, spend credibility cautiously and primarily late.
    # The seller still observes realized quality and can keep empirical precision
    # above a declining floor without guessing the buyer's exact cutoff.
    if model["pos_high"] < 1 or response_rate < 0.42:
        return 0.0
    precision_after = ((2.5 + model["pos_high"]) /
                       (3.3 + model["pos_high"] + model["pos_low"] + 1.0))
    credibility_floor = 0.78 - 0.18 * t
    if precision_after < credibility_floor:
        return 0.0
    return clamp((0.06 + 0.30 * t**1.8) * response_rate, 0.0, 0.34)

def persuasion_strategy(game):
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    price = finite_float(state["product_price"])
    p = clamp(finite_float(state.get("p"), 0.5), 0.0, 1.0)
    round_no = max(1, int(state.get("round", 1)))
    total_rounds = max(round_no, int(state.get("total_rounds", round_no)))
    t = (round_no - 1) / max(1, total_rounds - 1)
    seller_view = action_type in {"seller_message", "seller_recommendation"}
    model = update_persuasion_memory(game, seller_view)

    if seller_view:
        quality = state.get("current_quality")
        recommend = quality == "high"
        if quality == "low":
            probability = seller_low_pool_probability(
                game, state, model, p, price, t, round_no, total_rounds
            )
            recommend = stable_unit(game, "v4-seller-pool") < probability
        if action_type == "seller_recommendation":
            return {"decision": "yes" if recommend else "no"}
        return {"message": action_message(
            "My recommendation: BUY this product." if recommend else
            "My recommendation: PASS on this product this round."
        )}

    v, u = finite_float(state["v"]), finite_float(state["u"])
    if price <= u:
        return {"decision": "yes"}
    if price > v:
        return {"decision": "no"}
    signal = signal_polarity(state.get("seller_message"))
    posterior = p if signal is None else smoothed_signal_precision(model, signal, p)
    expected_value = posterior * v + (1.0 - posterior) * u
    observations = (model["pos_high"] + model["pos_low"] if signal is True else
                    model["neg_high"] + model["neg_low"] if signal is False else 0.0)
    remaining_fraction = (total_rounds - round_no) / max(1, total_rounds)
    information_bonus = (0.010 * max(0.0, v - u) * remaining_fraction /
                         math.sqrt(1.0 + observations) if signal is True else 0.0)
    return {"decision": "yes" if expected_value + information_bonus >= price else "no"}


## Validated dispatcher and safe fallback

Invalid moves and turn timeouts are scored at the fifth percentile. Every V3 move is
therefore validated locally; unexpected schemas fall back to a conservative legal
action and are recorded in `DECISION_LOG`.



In [ ]:
STRATEGIES = {
    "bargaining": bargaining_strategy,
    "negotiation": negotiation_strategy,
    "persuasion": persuasion_strategy,
}

def fallback_action(game):
    family = game["game_family"]
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    if family == "bargaining":
        if action_type == "offer":
            money = finite_float(state["money_to_divide"])
            alice = round(money / 2.0, 8)
            return {"alice_gain": alice, "bob_gain": money - alice}
        return {"decision": "accept"}
    if family == "negotiation":
        me = canonical_player(game.get("your_player", state["current_player"]))
        role = state[f"{me}_role"]
        value = finite_float(state[f"{me}_value"])
        if action_type == "offer":
            return {"product_price": max(0.0, value)}
        price = finite_float((state.get("last_offer") or {}).get("price"), value)
        profitable = price >= value if role == "seller" else price <= value
        if profitable:
            return {"decision": "AcceptOffer"}
        if final_round(state):
            return {"decision": "RejectOffer"}
        return {"decision": "RejectOffer", "product_price": max(0.0, value)}
    if action_type == "seller_message":
        return {"message": "My recommendation: PASS this round."}
    if action_type == "seller_recommendation":
        return {"decision": "no"}
    p = finite_float(state.get("p"), 0.5)
    expected = p * finite_float(state.get("v")) + (1 - p) * finite_float(state.get("u"))
    return {"decision": "yes" if expected >= finite_float(state["product_price"]) else "no"}

def validate_action(game, action):
    if not isinstance(action, dict):
        raise ValueError("strategy must return a dict")
    family = game["game_family"]
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    if family == "bargaining" and action_type == "offer":
        alice = finite_float(action["alice_gain"], math.nan)
        bob = finite_float(action["bob_gain"], math.nan)
        pot = finite_float(state["money_to_divide"])
        if not all(math.isfinite(x) and x >= 0 for x in (alice, bob)):
            raise ValueError("invalid bargaining allocation")
        if not math.isclose(alice + bob, pot, rel_tol=1e-10, abs_tol=1e-7):
            raise ValueError("bargaining gains do not sum to the pot")
    elif family == "bargaining":
        if action.get("decision") not in {"accept", "reject", "walkaway"}:
            raise ValueError("invalid bargaining decision")
    elif family == "negotiation" and action_type == "offer":
        if finite_float(action.get("product_price"), -1) < 0:
            raise ValueError("invalid negotiation price")
    elif family == "negotiation":
        if action.get("decision") not in {"AcceptOffer", "RejectOffer", "WalkAway"}:
            raise ValueError("invalid negotiation decision")
        if action["decision"] == "RejectOffer" and not final_round(state):
            if finite_float(action.get("product_price"), -1) < 0:
                raise ValueError("counteroffer required")
    elif action_type == "seller_message":
        if not isinstance(action.get("message"), str) or len(action["message"]) > 2000:
            raise ValueError("invalid persuasion message")
    elif action.get("decision") not in {"yes", "no"}:
        raise ValueError("invalid persuasion decision")
    if "message" in action and len(str(action["message"])) > 2000:
        raise ValueError("message exceeds 2,000 characters")
    return action

def strategy(game):
    error = None
    try:
        family = game["game_family"]
        action = validate_action(game, STRATEGIES[family](game))
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        action = validate_action(game, fallback_action(game))
        print(f"SAFE FALLBACK {game.get('game_id')}: {error}")
    with LOCK:
        DECISION_LOG.append({
            "game_id": game.get("game_id"), "family": game.get("game_family"),
            "round": (game.get("game_state") or {}).get("round"),
            "action": dict(action), "error": error,
        })
    return action



## Offline schema and history-regression tests

These tests make no API calls. In addition to the V3 schema boundaries, they reconstruct the observed extreme-discount bargaining configuration and repeated cycle, verify negotiation stall termination without violating individual rationality, and confirm that persuasion memory is separated by opponent role.


In [ ]:
def base_game(family, action_type, state, player="player_1", game_id="test",
              opponent=None):
    return {
        "game_id": game_id, "game_family": family, "your_player": player,
        "opponent": opponent or {"type": "hidden", "name": None},
        "valid_actions": {"type": action_type, "fields": {}},
        "game_state": state,
    }

def repeated_bargaining_history(cycles=4):
    history = []
    for index in range(cycles):
        history.append({"round": 2 * index + 1, "proposer": "player_1",
                        "offer": {"player_1_gain": 35, "player_2_gain": 65},
                        "decision": "reject"})
        history.append({"round": 2 * index + 2, "proposer": "player_2",
                        "offer": {"player_1_gain": 0.01, "player_2_gain": 99.99},
                        "decision": "reject"})
    return history

def repeated_negotiation_history(price_seller, price_buyer, cycles=4):
    return [{"round": i + 1,
             "offer": {"price": price_seller, "from_player": "player_1"},
             "decision": "RejectOffer", "decided_by": "player_2",
             "counteroffer": {"price": price_buyer, "from_player": "player_2"}}
            for i in range(cycles)]

def run_smoke_tests():
    for player in ("player_1", "player_2"):
        state = {"current_player": player, "round": 1, "max_rounds": 5,
                 "horizon_known": True, "money_to_divide": 100,
                 "delta_1": 0.9, "delta_2": 0.95,
                 "complete_information": True, "history": [],
                 "messages_allowed": True}
        action = strategy(base_game("bargaining", "offer", state, player,
                                    f"test-b-{player}"))
        assert math.isclose(action["alice_gain"] + action["bob_gain"], 100)

    # Regression for the observed 10% versus 0% inflation failure. The visible
    # equilibrium requires giving the almost-undiscounted responder over 90%.
    extreme = {"current_player": "player_1", "round": 1,
               "horizon_known": False, "money_to_divide": 100,
               "delta_1": 0.9, "delta_2": 0.999,
               "complete_information": True, "history": []}
    extreme_action = strategy(base_game("bargaining", "offer", extreme,
                                        "player_1", "test-b-extreme"))
    assert extreme_action["bob_gain"] > 90

    cycle_state = dict(extreme, current_player="player_1", round=9,
                       history=repeated_bargaining_history(),
                       last_offer={"player_1_gain": 0.01, "player_2_gain": 99.99})
    assert strategy(base_game("bargaining", "decision", cycle_state,
                              "player_1", "test-b-cycle"))["decision"] == "accept"

    full_negotiation = {"current_player": "player_1",
        "player_1_role": "seller", "player_2_role": "buyer",
        "player_1_value": 40, "player_2_value": 100,
        "complete_information": True, "round": 1, "max_rounds": 5,
        "horizon_known": True, "history": [], "messages_allowed": True}
    price = strategy(base_game("negotiation", "offer", full_negotiation,
                               "player_1", "test-n-full"))["product_price"]
    assert 40 <= price <= 100

    # Repeated 101/100 prices are infeasible for a buyer valued at 100. V4 exits
    # safely rather than polling through 99 rounds or accepting negative utility.
    stalled = {"current_player": "player_2", "player_1_role": "seller",
        "player_2_role": "buyer", "player_2_value": 100,
        "complete_information": False, "round": 9, "horizon_known": False,
        "last_offer": {"price": 101, "from_player": "player_1"},
        "history": repeated_negotiation_history(101, 100)}
    assert strategy(base_game("negotiation", "decision", stalled,
                              "player_2", "test-n-stall"))["decision"] == "WalkAway"
    stalled_at_value = dict(stalled, last_offer={"price": 100, "from_player": "player_1"},
                            history=repeated_negotiation_history(100, 100))
    assert strategy(base_game("negotiation", "decision", stalled_at_value,
                              "player_2", "test-n-zero"))["decision"] == "AcceptOffer"

    high_product = {"current_quality": "high", "product_price": 50, "p": 0.5,
                    "v": 100, "u": 0, "round": 1, "total_rounds": 10,
                    "history": []}
    assert strategy(base_game("persuasion", "seller_recommendation", high_product,
                              "player_1", "test-p-high")) == {"decision": "yes"}
    expensive = {"seller_message": "I recommend buying this product.",
                 "product_price": 101, "p": 0.9, "v": 100, "u": 0,
                 "round": 1, "total_rounds": 5, "history": []}
    assert strategy(base_game("persuasion", "buyer_decision", expensive,
                              "player_2", "test-p-expensive")) == {"decision": "no"}

    # The same named opponent receives separate memory namespaces by role.
    named = {"type": "agent", "name": "same-opponent"}
    seller_history = dict(high_product, current_quality="low", round=3,
        history=[{"round": 1, "seller_message": {"decision": "yes"},
                  "buyer_decision": "yes", "quality": "high"}])
    strategy(base_game("persuasion", "seller_recommendation", seller_history,
                       "player_1", "test-p-role-a", named))
    buyer_history = {"seller_message": {"decision": "yes"}, "product_price": 40,
        "p": 0.5, "v": 100, "u": 0, "round": 2, "total_rounds": 5,
        "history": [{"round": 1, "seller_message": {"decision": "yes"},
                     "buyer_decision": "yes", "quality": "low"}]}
    strategy(base_game("persuasion", "buyer_decision", buyer_history,
                       "player_2", "test-p-role-b", named))
    buyer_key = opponent_key(base_game("persuasion", "seller_recommendation",
                              seller_history, "player_1", "x", named),
                             "persuasion:buyer-response")
    reliability_key = opponent_key(base_game("persuasion", "buyer_decision",
                                    buyer_history, "player_2", "y", named),
                                   "persuasion:seller-reliability")
    assert buyer_key != reliability_key
    assert PERSUASION_MEMORY[buyer_key]["pos_high"] == 1
    assert PERSUASION_MEMORY[reliability_key]["pos_low"] == 1

    errors = [entry for entry in DECISION_LOG if entry["error"]]
    assert not errors, errors
    print("All V4 schema and history-regression tests passed.")

run_smoke_tests()


## Controlled live evaluation — one family at a time

Do not immediately replace V3 across all queues. Start with 15–25 games in one family, record `client.stats()` before and after, inspect fallback and cycle decisions, then compare against a similar V3 batch. The default below tests bargaining because V4's most important regression fix is the observed 99-round bargaining failure. Change `EVALUATION_FAMILY` only after reviewing the first batch.

The SDK drains in-flight games before returning. Keep one client per API key and begin at concurrency one; raise it only after a clean bounded run.


In [ ]:
from glee_sdk import GleeClient

EVALUATION_FAMILY = "bargaining"  # then "negotiation", then "persuasion"
CONCURRENCY = 1
MAX_GAMES = 20
MAX_TIME = 3600

client = GleeClient(api_key=os.environ["GLEE_API_KEY"])
before = client.stats()
print("Before:", before)
client.run(
    strategy,
    game_families=[EVALUATION_FAMILY],
    concurrency=CONCURRENCY,
    max_games=MAX_GAMES,
    max_time=MAX_TIME,
)
after = client.stats()
print("After:", after)


## Inspect decisions, fallbacks, and model state


In [ ]:
fallbacks = [entry for entry in DECISION_LOG if entry["error"]]
print("Fallback count:", len(fallbacks))
print("Recent decisions:")
display(list(DECISION_LOG)[-20:])

print("Named/game-local bargaining profiles:", len(BARGAINING_MEMORY))
print("Named/game-local negotiation profiles:", len(NEGOTIATION_MEMORY))
print("Role-separated persuasion profiles:", len(PERSUASION_MEMORY))
